<a href="https://colab.research.google.com/github/arildbn/bban4040/blob/main/martra-notebooks/6-3_aws-bedrock-nl2sql-client.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div>
    <h1>Large Language Models Projects</a></h1>
    <h3>Apply and Implement Strategies for Large Language Models</h3>
    <h2>6.3-Calling AWS Bedrock from Python. </h2>
    <h3></h3>
    <p>by <b>Pere Martra</b></p>
</div>



In this notebook, we make a call to a Model from AWS Bedrock  that we've set up to work as a translator for SQL queries from natural language.


In [ ]:
%pip install -q boto3

In [ ]:
# === portable-setup (bban4040) ===
# Secrets resolve from Colab "Secrets" (userdata) on Colab, or environment
# variables / a local .env file when running locally. Nothing is hardcoded.
import os
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass


def get_secret(name, default=None):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value.strip()
    except Exception:
        pass
    value = os.environ.get(name, default)
    return value.strip() if isinstance(value, str) else value


_key = get_secret("OPENAI_API_KEY")
if _key:
    os.environ["OPENAI_API_KEY"] = _key


# AWS credentials for Bedrock. On Colab set these in "Secrets"; locally in .env
# or environment variables. Region defaults to us-west-2 (Bedrock model access
# must be enabled for the chosen model in that region).
for _name in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"):
    _v = get_secret(_name)
    if _v:
        os.environ[_name] = _v
_region = get_secret("AWS_DEFAULT_REGION") or get_secret("AWS_REGION")
os.environ["AWS_DEFAULT_REGION"] = _region or "us-west-2"


In [ ]:
import boto3
import json
from getpass import getpass

In [ ]:
# AWS credentials for Bedrock. We never prompt, so the notebook runs unattended.
# Set AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY (and AWS_DEFAULT_REGION) in Colab
# Secrets or .env, and request Bedrock model access, to enable the live calls.
aws_access_key_id = os.environ.get("AWS_ACCESS_KEY_ID")

In [ ]:
aws_secret_access_key = os.environ.get("AWS_SECRET_ACCESS_KEY")
AWS_AVAILABLE = bool(aws_access_key_id) and bool(aws_secret_access_key)
if AWS_AVAILABLE:
    print("AWS credentials found - live Bedrock calls will run.")
else:
    print("No AWS credentials - skipping live Bedrock calls. "
          "Set AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY to enable.")

In [ ]:
client = None
if AWS_AVAILABLE:
    client = boto3.client("bedrock-runtime",
                          region_name=os.getenv("AWS_DEFAULT_REGION", "us-west-2"),
                          aws_access_key_id=aws_access_key_id,
                          aws_secret_access_key=aws_secret_access_key)


In [ ]:
# Set the model ID, e.g., Llama 3 8B Instruct.
model_id = "meta.llama3-8b-instruct-v1:0"

In [ ]:
# Define the user message to send.
user_message = "What is the name of the best paid employee?"

In [ ]:
model_instructions = """
Your task is to convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word to appropriately answer the question.
- **Return Only SQL Code. 
   ### Input
   Generate a SQL query that answers the question below.
   This query will run on a database whose schema is represented in this string:

   create table employees(
       ID_Usr INT primary key,-- Unique Id for employee
       name VARCHAR -- Name of employee
       );

   create table salary(
       ID_Usr INT,-- Unique Id for employee
       year DATE, -- Date
       salary FLOAT, --Salary of employee
       foreign key (ID_Usr) references employees(ID_Usr) -- Join Employees with salary
       );

   create table studies(
       ID_study INT, -- Unique ID study
       ID_Usr INT, -- ID employee
       educational_level INT,  -- 5=phd, 4=Master, 3=Bachelor
       Institution VARCHAR, --Name of instituon where eployee studied
       Years DATE, -- Date acomplishement stdy
       Speciality VARCHAR, -- Speciality of studies
       primary key (ID_study, ID_Usr), --Primary Key ID_Usr + ID_Study
       foreign key(ID_Usr) references employees (ID_Usr)
       );

"""


In [ ]:
# Embed the message in Llama 3's prompt format.
prompt = f"""
<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>
{model_instructions}
<|eot_id|>
<|start_header_id|>user<|end_header_id|>
{user_message}
<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
"""

In [ ]:
print (prompt)

In [ ]:
# Format the request payload using the model's native structure.
hyper = {
    "prompt": prompt,
    # Optional inference parameters:
    "max_gen_len": 512,
    "temperature": 0.0
}

In [ ]:
if AWS_AVAILABLE:
    # Encode and send the request.
    response = client.invoke_model(body=json.dumps(hyper), modelId=model_id)

In [ ]:
if AWS_AVAILABLE:
    # Decode the native response body.
    model_response = json.loads(response["body"].read())

In [ ]:
if AWS_AVAILABLE:
    response_text = model_response["generation"]
    print(response_text)
else:
    print("skipped: no AWS credentials")